# Olist E-Commerce Data Quality Audit

This notebook evaluates the structure, quality, and key relationships of the Olist e-commerce datasets before downstream SQL and customer analytics.

The audit focuses on dataset structure, key integrity, missing values, duplicate records, and data characteristics that could affect revenue, customer, retention, product, payment, review, and seller analysis.


## 1. Data Loading

Load the seven core Olist datasets used in the project and create a dictionary so the same validation logic can be applied consistently across tables.


In [1]:
import pandas as pd


In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers
}


## 2. Dataset Structure

Start with a compact table-level overview, then inspect a small sample of the customer table to confirm that the data loaded as expected.


In [3]:
dataset_overview = pd.DataFrame(
    {
        "rows": {name: df.shape[0] for name, df in datasets.items()},
        "columns": {name: df.shape[1] for name, df in datasets.items()},
        "duplicate_rows": {name: df.duplicated().sum() for name, df in datasets.items()}
    }
)

dataset_overview


,rows,columns,duplicate_rows
customers,99441,5,0
orders,99441,8,0
order_items,112650,7,0
products,32951,9,0
payments,103886,5,0
reviews,99224,7,0
sellers,3095,4,0


In [4]:
customers.head()


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 3. General Data Quality Audit

Apply the same basic audit to every dataset: shape, data types, missing values, and fully duplicated rows. This provides a repeatable first-pass validation before table-specific checks.


In [5]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:", df.duplicated().sum())
    print("-" * 50)



CUSTOMERS
Shape: (99441, 5)

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows: 0
--------------------------------------------------

ORDERS
Shape: (99441, 8)

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_

## 4. Customer Identity and Key Integrity

The customer data contains both `customer_id` and `customer_unique_id`. These identifiers serve different purposes, so their uniqueness must be checked before customer-level analysis.


In [6]:
print("Rows in customers:", len(customers))
print("Unique customer_id values:", customers["customer_id"].nunique())
print("Unique customer_unique_id values:", customers["customer_unique_id"].nunique())


Rows in customers: 99441
Unique customer_id values: 99441
Unique customer_unique_id values: 96096


In [7]:
customers["customer_unique_id"].value_counts().head(10)


customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
de34b16117594161a6a89c50b289d35a     6
Name: count, dtype: int64

**Interpretation:** `customer_id` is unique at the customer-record level, while `customer_unique_id` can appear across multiple records. Therefore, `customer_unique_id` is the appropriate identifier for repeat-purchase and retention analysis, while `customer_id` is used to join customers to orders.


## 5. Order Lifecycle and Missingness

Inspect order status distribution and investigate whether missing delivery timestamps are associated with incomplete order lifecycles.


In [8]:
orders["order_status"].value_counts()


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
print("Rows in orders:", len(orders))
print("Unique order_id values:", orders["order_id"].nunique())


Rows in orders: 99441
Unique order_id values: 99441


In [10]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

**Interpretation:** Missing delivery-related timestamps are concentrated among orders that had not completed the delivery lifecycle, such as shipped, canceled, unavailable, invoiced, or processing orders. A small number of delivered orders with missing delivery dates should be treated as anomalies.


## 6. Table-Level Key and Relationship Checks

Validate the keys used to identify rows and connect tables. Composite keys are required where a single identifier can legitimately appear more than once.


In [11]:
key_checks = pd.Series(
    {
        "order_items duplicate (order_id, order_item_id)": order_items.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum(),
        "distinct order_ids in order_items": order_items["order_id"].nunique(),
        "distinct product_ids in order_items": order_items["product_id"].nunique(),
        "products rows": products.shape[0],
        "unique product_id values": products["product_id"].nunique(),
        "payments duplicate (order_id, payment_sequential)": payments.duplicated(
            subset=["order_id", "payment_sequential"]
        ).sum(),
        "reviews rows": reviews.shape[0],
        "unique review_id values": reviews["review_id"].nunique(),
        "unique order_id values in reviews": reviews["order_id"].nunique(),
        "sellers rows": sellers.shape[0],
        "unique seller_id values": sellers["seller_id"].nunique()
    }
)

key_checks


order_items duplicate (order_id, order_item_id)          0
distinct order_ids in order_items                    98666
distinct product_ids in order_items                  32951
products rows                                        32951
unique product_id values                             32951
payments duplicate (order_id, payment_sequential)        0
reviews rows                                         99224
unique review_id values                              98410
unique order_id values in reviews                    98673
sellers rows                                          3095
unique seller_id values                               3095
dtype: int64

**Interpretation:** `order_id + order_item_id` uniquely identifies order-item rows, and `order_id + payment_sequential` uniquely identifies payment rows. `product_id` and `seller_id` are unique in their entity tables. Review records require additional care because neither `review_id` nor `order_id` is unique at the row level.


## 7. Business-Relevant Data Characteristics

Review distributions and missing metadata that may influence downstream business analysis.


In [12]:
order_items[["price", "freight_value"]].describe()


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [13]:
products.isna().sum()


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [14]:
payments["payment_type"].value_counts()


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [15]:
reviews["review_score"].value_counts().sort_index()


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [16]:
sellers["seller_state"].value_counts()


seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [17]:
customers["customer_city"].value_counts().head(10)


customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64

## 8. Audit Summary and Analysis Implications

- `customers`: 99,441 rows with no missing or fully duplicated rows. `customer_id` is unique, while `customer_unique_id` repeats across some records, so `customer_unique_id` should be used for repeat-customer and retention analysis.
- `orders`: 99,441 rows with unique `order_id` values and no fully duplicated rows. Missing delivery-related timestamps are mostly associated with non-delivered order statuses, suggesting that the missingness is largely consistent with incomplete order lifecycles.
- `order_items`: `order_id + order_item_id` is unique at the row level. The table contains 98,666 distinct orders and 32,951 distinct sold products. Price and freight values are right-skewed, with some high-value observations.
- `products`: `product_id` is unique. Some product metadata is missing, especially category/content fields, while physical measurements are nearly complete.
- `payments`: `order_id + payment_sequential` is unique. Credit cards are the dominant payment method, followed by boleto.
- `reviews`: review scores are concentrated at 4 and 5 stars. Neither `review_id` nor `order_id` is unique at the row level, so repeated review records require care in later joins.
- `sellers`: 3,095 sellers with unique `seller_id` values. Sellers are heavily concentrated in São Paulo state.

### Downstream analysis decisions

- Use `customer_unique_id` for customer-level repeat-purchase, retention, and customer-value analysis.
- Use `customer_id` to join `customers` to `orders`.
- Treat missing delivery timestamps in the context of order status rather than assuming all missing values are data errors.
- Account for the fact that not every order appears in `order_items` when interpreting analyses based on order-item joins.
- Use caution when joining review records because review and order identifiers are not unique at the row level.
- Consider medians alongside means when describing highly skewed price and freight distributions.

Overall, the data is sufficiently structured for downstream customer and revenue analysis, provided these identity, missingness, and relationship considerations are accounted for.
